# Q2 action risk delta support

Frozen read-only preflight; see `plan.json`.

In [1]:
import hashlib, json, os
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import numpy as np
import pandas as pd
import qlib
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd(); card=root/'research/stage1_q2_action_risk_delta_support_20261112'
plan=json.loads((card/'plan.json').read_text())
source=root/plan['source_card']
qlib.init(provider_uri=str(root/'data/qlib/stage1_event_guard_fullminute_20261028'),region='us',kernels=1)
sha=lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(source/'summary.json')==plan['source_summary_sha256']
assert sha(source/'verification_manifest.json')==plan['source_manifest_sha256']
objects,receipt=load_verified_recorder_objects(source,expected_recorder_id=plan['source_recorder_id'],artifacts=plan['read_only_artifacts'],expected_manifest_sha256=plan['source_manifest_sha256'])
rows=objects['row_diagnostics'].copy()
assert len(rows)==objects['report']['risk_rows']==7485
assert set(pd.to_datetime(rows.day).dt.year)=={2019,2020}
assert rows[['actual_risk','hold_actual_risk','true_action_delta','predicted_action_delta']].notna().all().all()
assert np.max(np.abs(rows.true_action_delta-(rows.actual_risk-rows.hold_actual_risk)))<1e-12
rows['day']=pd.to_datetime(rows.day).dt.normalize()
rows['large_delta']=rows.true_action_delta.abs()>0.0001
rows['nonzero_delta']=rows.true_action_delta.abs()>1e-12
annual={}
for year in (2019,2020):
    group=rows.loc[rows.day.dt.year==year]
    day_large=group.groupby('day').large_delta.any()
    day_nonzero=group.groupby('day').nonzero_delta.any()
    day_summary=group.groupby('day')[['true_action_delta','predicted_action_delta']].mean()
    annual[str(year)]={
      'rows':int(len(group)), 'days':int(group.day.nunique()),
      'days_large_1bp':int(day_large.sum()), 'days_nonzero':int(day_nonzero.sum()),
      'large_1bp_day_fraction':float(day_large.mean()),
      'exact_zero_row_fraction':float((~group.nonzero_delta).mean()),
      'true_delta_day_mean':float(day_summary.true_action_delta.mean()),
      'predicted_delta_day_mean':float(day_summary.predicted_action_delta.mean()),
      'predicted_minus_actual_day_mean':float((day_summary.predicted_action_delta-day_summary.true_action_delta).mean()),
      'true_delta_abs_p50':float(group.true_action_delta.abs().quantile(.5)),
      'true_delta_abs_p90':float(group.true_action_delta.abs().quantile(.9)),
      'delta_row_spearman':float(group[['true_action_delta','predicted_action_delta']].corr(method='spearman').iloc[0,1])}
support_pass=all(v['days_large_1bp']>=20 and v['days_nonzero']>=20 for v in annual.values())
summary={'experiment':plan['experiment'],'decision':'risk_action_delta_support_pass_only' if support_pass else 'risk_action_delta_support_failed',
 'historical_research_passed':False,'new_model_fits':0,'new_native_accounts':0,
 'source_recorder_id':plan['source_recorder_id'],'source_manifest_sha256':plan['source_manifest_sha256'],
 'source_readback':receipt,'annual':annual,'support_pass':support_pass,
 'limitations':['Reused 2019-2020 in-sample optimization held actions, not temporal OOS or SOXX evidence','Difference of conditional 95pct quantile predictions is not the conditional mean or 95pct quantile of realized action-risk differences; mean comparison is not calibration evidence','No causal inference or fitted residual risk model'],
 'artifact_policy':{'policy':'slim_card_recorder_first','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False)+'\n')
print(json.dumps({'decision':summary['decision'],'annual':annual},indent=2))

{
  "decision": "risk_action_delta_support_pass_only",
  "annual": {
    "2019": {
      "rows": 3780,
      "days": 252,
      "days_large_1bp": 252,
      "days_nonzero": 252,
      "large_1bp_day_fraction": 1.0,
      "exact_zero_row_fraction": 0.07592592592592592,
      "true_delta_day_mean": 0.001123458122499404,
      "predicted_delta_day_mean": 0.00435006424307476,
      "predicted_minus_actual_day_mean": 0.003226606120575355,
      "true_delta_abs_p50": 0.000503400160386358,
      "true_delta_abs_p90": 0.009822060667685806,
      "delta_row_spearman": 0.3640172684629741
    },
    "2020": {
      "rows": 3705,
      "days": 247,
      "days_large_1bp": 246,
      "days_nonzero": 247,
      "large_1bp_day_fraction": 0.9959514170040485,
      "exact_zero_row_fraction": 0.08124156545209177,
      "true_delta_day_mean": 0.00017223798308042696,
      "predicted_delta_day_mean": 0.005197655281895326,
      "predicted_minus_actual_day_mean": 0.005025417298814901,
      "true_delta_abs

The executed summary is in `summary.json`. This preflight does not establish SOXX excess.